# Prior structure visualization

This notebook visualizes the learned auditory prior with PCA projections and score-field diagnostics.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA



## Inputs
Expected to be available in memory or loaded from disk:
- `score_fn`: trained score model
- `X_clean`: natural-sound embeddings `[N, D]`


In [ ]:
# Example placeholders
# score_fn = ...
# X_clean = ...



In [ ]:
# PCA projection
pca = PCA(n_components=2)
X2 = pca.fit_transform(X_clean)
plt.figure(figsize=(6, 5))
plt.scatter(X2[:, 0], X2[:, 1], s=5, alpha=0.3)
plt.title('Natural-sound embeddings (PCA)')
plt.show()



In [ ]:
# Score field on PCA grid
mins = X2.min(0)
maxs = X2.max(0)
xx, yy = np.meshgrid(np.linspace(mins[0], maxs[0], 25), np.linspace(mins[1], maxs[1], 25))
grid2 = np.stack([xx.ravel(), yy.ravel()], axis=1)
gridD = pca.inverse_transform(grid2)
with torch.no_grad():
    g = torch.as_tensor(gridD, dtype=torch.float32, device=next(score_fn.parameters()).device)
    s = score_fn.forward(g).detach().cpu().numpy().reshape(len(gridD), -1)
s2 = s @ pca.components_.T
plt.figure(figsize=(6, 5))
plt.scatter(X2[:,0], X2[:,1], s=5, alpha=0.2)
plt.quiver(grid2[:,0], grid2[:,1], s2[:,0], s2[:,1], alpha=0.7)
plt.title('Projected score field')
plt.show()

